# 04 — Supervised baselines
**Level: intermediate → advanced.** Compare simple, explainable models first. This notebook does not overwrite the frozen held-out test set, and only reports validation performance.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from nohack.features import FEATURE_NAMES
from nohack.models.hybrid import encode_flow
root = Path(os.environ.get("NOHACK_SPLITS", "data/processed/nohack_splits"))
train, val = (pd.read_parquet(root / f"{name}.parquet") for name in ("train", "val"))
xtr, xval = encode_flow(train[list(FEATURE_NAMES)]), encode_flow(val[list(FEATURE_NAMES)])
ytr, yval = train.label.str.upper().ne("BENIGN").astype(int), val.label.str.upper().ne("BENIGN").astype(int)
print("Rows:", len(ytr), len(yval))

In [ ]:
models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500, class_weight="balanced")),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=42, n_jobs=-1),
}
results = []
for name, model in models.items():
    model.fit(xtr, ytr)
    prob = model.predict_proba(xval)[:, 1]
    results.append({"model": name, "validation_ap": average_precision_score(yval, prob),
                    "validation_auc": roc_auc_score(yval, prob),
                    "validation_f1_at_0.5": f1_score(yval, prob >= 0.5)})
print(pd.DataFrame(results).round(4).to_string(index=False))

### Think like a data scientist
An accuracy number alone is insufficient for a rare-attack setting. Compare precision, recall, PR-AUC, false positives per legitimate flows and robustness on new networks. Keep the test split sealed until model selection is complete.